# Head-versus-body pruning ablation

Magnitude pruning in the compression matrix removes weights from every linear and convolutional layer,
including the 4,386-parameter classification head, and the decision-layer remedy re-fits a fresh dense
head. The results so far are therefore compatible with two attributions: the collapse reflects damage to
the network body (with the head merely re-reading a changed representation), or the collapse follows
directly from deleting 80% of the head's own weights (and recovery follows from restoring them). This
notebook separates the two.

**Condition A, body-only pruning.** Every linear and convolutional layer except the classification head
is pruned to 80% per-layer sparsity; the head stays dense. Evaluated one-shot and after the standard
fine-tune. If the per-class collapse persists with a fully dense head, head-weight deletion is not
necessary for the collapse.

**Condition B, head-only pruning.** Only the classification head is pruned to 80%; the body stays dense.
Evaluated one-shot and after the standard fine-tune. If little collapse appears, head-weight deletion is
not sufficient for it either.

Both conditions reuse the pipeline's own `prune_and_finetune` unchanged, with only the prune step
substituted, so fine-tuning, mask re-application, evaluation, and the 2-sigma collapse criterion on the
thirteen measurable classes are identical to the main experiments. Global sparsity differs by design
(body-only at per-layer 80% is roughly 68% global; head-only is roughly 12%), which is reported alongside.
Run on a GPU runtime.

In [ ]:
# --- Colab bootstrap ---
try:
    from google.colab import drive; drive.mount('/content/drive')
    REPO = '/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression'
except Exception:
    REPO = '.'
import os, sys
os.chdir(REPO); sys.path.insert(0, REPO)

import copy
import numpy as np, pandas as pd, torch
import torch.nn as nn
import torch.nn.utils.prune as tprune
from src.config import CFG, PATHS, set_all_seeds
from src import data as D, models as M, compression as C, train as TR

SEED = CFG['anchor_seed']
set_all_seeds(SEED)
ARCH = 'cnn1d'; DATASET = 'ciciot2023'
print('anchor seed:', SEED)

In [ ]:
import torch
print('device:', 'cuda (' + torch.cuda.get_device_name(0) + ')' if torch.cuda.is_available() else 'CPU (SLOW)')
assert torch.cuda.is_available(), \
    'CPU runtime detected. Switch to a T4 GPU (Runtime -> Change runtime type) before running.'

In [ ]:
# Load the dataset, the exact primary split, and the anchor checkpoint
df = D.clean(D.load_raw(DATASET, subsample=True, seed=SEED), DATASET)
splits = D.temporal_within_capture_split(df, seed=SEED)
feat_cols = TR.feature_columns(df)

from sklearn.preprocessing import LabelEncoder, StandardScaler
le = LabelEncoder().fit(df['label'].to_numpy())
scaler = StandardScaler().fit(df.loc[splits['train'], feat_cols].to_numpy(np.float32))
n_classes = len(le.classes_)

ck = torch.load(PATHS.model(DATASET, ARCH, 'M0', SEED), map_location='cpu', weights_only=False)
sd = ck['state_dict']
ch = (int(sd['conv.0.weight'].shape[0]), int(sd['conv.3.weight'].shape[0]))
anchor = M.build(ARCH, len(feat_cols), n_classes, channels=ch)
anchor.load_state_dict(sd); anchor = anchor.to(C.DEVICE).eval()
print('anchor loaded; head is', anchor.head)

In [ ]:
# Collapse criterion: paper's thirteen measurable classes, 2-sigma null band
UNSTABLE = {'DoS-TCP_Flood', 'DDoS-SynonymousIP_Flood', 'DDoS-TCP_Flood', 'DDoS-SYN_Flood'}
nbnd = pd.read_csv(PATHS.tables('baseline', 'cnn1d_M0_null_band_5seed.csv')).set_index('label')
measurable = [c for c in nbnd.index[nbnd['tier'] == 'measurable'] if c not in UNSTABLE]
assert len(measurable) == 13
band = {c: (nbnd.loc[c, 'mean'], nbnd.loc[c, 'null_band_2sigma']) for c in measurable}

def collapsed_classes(rec_by_label):
    out = []
    for c in measurable:
        m, thr = band[c]
        r = rec_by_label.get(c, float('nan'))
        if not np.isnan(r) and (m - r) > thr:
            out.append(c)
    return out

@torch.no_grad()
def recall_by_label_for_model(model):
    entry = {'model': model, 'is_half': False, 'is_int8': False}
    rec_idx, macro, _, _, _ = C.evaluate_cell(entry, df, splits, le, scaler, feat_cols, which='test')
    return {le.classes_[ci]: v for ci, v in rec_idx.items()}, macro

def sparsity_report(model):
    tot = sum(p.numel() for p in model.parameters())
    nz = sum(int((p != 0).sum()) for p in model.parameters())
    return 1 - nz / tot

In [ ]:
# Scoped pruning: prune every Linear/Conv1d EXCEPT (or ONLY) the classification head.
def scoped_magnitude_prune(model, amount, scope):
    m = copy.deepcopy(model)
    head = m.head
    for module in m.modules():
        if not isinstance(module, (nn.Linear, nn.Conv1d)):
            continue
        is_head = module is head
        if (scope == 'body_only' and is_head) or (scope == 'head_only' and not is_head):
            continue
        tprune.l1_unstructured(module, name='weight', amount=amount)
        tprune.remove(module, 'weight')
    return m

def make_pruner(scope):
    def _pruner(model, amount):
        return scoped_magnitude_prune(model, amount, scope)
    return _pruner

# Baseline reference on this anchor
rec_M0, mf1_M0 = recall_by_label_for_model(anchor)
print(f'M0 macro-F1 = {mf1_M0:.4f}')

In [ ]:
# Run both conditions: one-shot and pipeline fine-tuned (prune step substituted, all else identical)
results = []
for scope in ['body_only', 'head_only']:
    set_all_seeds(SEED)
    m_os = scoped_magnitude_prune(anchor, 0.80, scope).to(C.DEVICE).eval()
    rec_os, mf1_os = recall_by_label_for_model(m_os)
    coll_os = collapsed_classes(rec_os)
    sp_os = sparsity_report(m_os)
    print(f'{scope} one-shot: macro-F1 {mf1_os:.4f} | global sparsity {sp_os:.3f} | collapsed {len(coll_os)}/13')
    print('   ', coll_os)
    results.append({'condition': scope, 'finetuned': False, 'macro_f1': round(mf1_os, 4),
                    'global_sparsity': round(sp_os, 4), 'n_collapsed_of_13': len(coll_os),
                    'collapsed': ';'.join(coll_os)})

    set_all_seeds(SEED)
    orig = C._magnitude_prune
    C._magnitude_prune = make_pruner(scope)
    try:
        m_ft, _l, _s = C.prune_and_finetune(anchor, df, DATASET, splits, SEED, 0.80,
                                            arch=ARCH, verbose=False)
    finally:
        C._magnitude_prune = orig
    m_ft = m_ft.to(C.DEVICE).eval()
    rec_ft, mf1_ft = recall_by_label_for_model(m_ft)
    coll_ft = collapsed_classes(rec_ft)
    sp_ft = sparsity_report(m_ft)
    print(f'{scope} + finetune: macro-F1 {mf1_ft:.4f} | global sparsity {sp_ft:.3f} | collapsed {len(coll_ft)}/13')
    print('   ', coll_ft)
    results.append({'condition': scope, 'finetuned': True, 'macro_f1': round(mf1_ft, 4),
                    'global_sparsity': round(sp_ft, 4), 'n_collapsed_of_13': len(coll_ft),
                    'collapsed': ';'.join(coll_ft)})
    del m_os, m_ft; torch.cuda.empty_cache()

In [ ]:
# Reference rows (from the main experiments) and summary
results.append({'condition': 'full_prune (reference)', 'finetuned': False, 'macro_f1': 0.0080,
                'global_sparsity': 0.7836, 'n_collapsed_of_13': 13, 'collapsed': ''})
results.append({'condition': 'full_prune (reference)', 'finetuned': True, 'macro_f1': 0.3377,
                'global_sparsity': 0.7836, 'n_collapsed_of_13': 10, 'collapsed': ''})
summary = pd.DataFrame(results)
print(summary.to_string(index=False))
out = PATHS.tables('explain', 'prune_scope_head_vs_body.csv')
summary.to_csv(out, index=False)
print()
print('saved:', out)

## Outputs

`prune_scope_head_vs_body.csv` reports, for body-only and head-only pruning at 80% (one-shot and
fine-tuned) alongside the full-prune reference, the test macro-F1, whole-model sparsity, and the number
of the thirteen measurable classes that collapse under the standard 2-sigma criterion. The attribution
reading is: substantial collapse under body-only pruning (dense head) shows head-weight deletion is not
necessary for the collapse; the absence of comparable collapse under head-only pruning (dense body) shows
it is not sufficient. The reverse pattern would attribute the collapse to the head itself.

In [ ]:
# --- End-of-unit discipline: commit + push (credentials restored from Drive) ---
import subprocess, shutil
DRIVE_ROOT = '/content/drive/MyDrive/IoT_Trust_Research'
for f in ['.gitconfig', '.git-credentials']:
    src = os.path.join(DRIVE_ROOT, f)
    if os.path.exists(src):
        shutil.copy(src, f'/root/{f}')
subprocess.run(['git', 'config', '--global', 'credential.helper', 'store'], check=True)
subprocess.run(['git', 'add', '-A'], check=True)
print(subprocess.run(['git', 'commit', '-m',
    'head-vs-body pruning scope ablation at 80% sparsity'], capture_output=True, text=True).stdout)
print(subprocess.run(['git', 'push'], capture_output=True, text=True).stderr or 'pushed')